In [1]:
import tensorflow as tf
from pathlib import Path
import matplotlib.pyplot as plt

In [2]:
BASE_DIR = Path("../datasets/classification")

TRAIN_DIR = BASE_DIR / "Training"

IMG_SIZE = (224,224)

BATCH_SIZE = 16

In [3]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

Found 5600 files belonging to 4 classes.
Using 4480 files for training.
Found 5600 files belonging to 4 classes.
Using 1120 files for validation.


In [4]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(AUTOTUNE)

val_ds = val_ds.prefetch(AUTOTUNE)

In [5]:
data_augmentation = tf.keras.Sequential([

    tf.keras.layers.RandomFlip("horizontal"),

    tf.keras.layers.RandomRotation(0.1),

    tf.keras.layers.RandomZoom(0.1)

])

In [6]:
base_model = tf.keras.applications.EfficientNetB0(

    include_top=False,

    weights="imagenet",

    input_shape=(224,224,3)

)

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 9s 1us/step


In [7]:
base_model.trainable = False

In [8]:
inputs = tf.keras.Input(
    shape=(224,224,3)
)

x = data_augmentation(inputs)

x = tf.keras.applications.efficientnet.preprocess_input(x)

x = base_model(x)

x = tf.keras.layers.GlobalAveragePooling2D()(x)

x = tf.keras.layers.Dropout(0.3)(x)

outputs = tf.keras.layers.Dense(
    4,
    activation="softmax"
)(x)

model = tf.keras.Model(
    inputs,
    outputs
)

In [9]:
model.compile(

    optimizer="adam",

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]

)

In [10]:
early_stop = tf.keras.callbacks.EarlyStopping(

    monitor="val_loss",

    patience=3,

    restore_best_weights=True

)

In [11]:
history = model.fit(

    train_ds,

    validation_data=val_ds,

    epochs=10,

    callbacks=[early_stop]

)

Epoch 1/10
280/280 ━━━━━━━━━━━━━━━━━━━━ 135s 450ms/step - accuracy: 0.7480 - loss: 0.6681 - val_accuracy: 0.8223 - val_loss: 0.4662
Epoch 2/10
280/280 ━━━━━━━━━━━━━━━━━━━━ 123s 438ms/step - accuracy: 0.8272 - loss: 0.4694 - val_accuracy: 0.8393 - val_loss: 0.4211
Epoch 3/10
280/280 ━━━━━━━━━━━━━━━━━━━━ 139s 427ms/step - accuracy: 0.8371 - loss: 0.4310 - val_accuracy: 0.8500 - val_loss: 0.3887
Epoch 4/10
280/280 ━━━━━━━━━━━━━━━━━━━━ 126s 449ms/step - accuracy: 0.8540 - loss: 0.3900 - val_accuracy: 0.8723 - val_loss: 0.3495
Epoch 5/10
280/280 ━━━━━━━━━━━━━━━━━━━━ 127s 454ms/step - accuracy: 0.8616 - loss: 0.3777 - val_accuracy: 0.8830 - val_loss: 0.3365
Epoch 6/10
280/280 ━━━━━━━━━━━━━━━━━━━━ 122s 434ms/step - accuracy: 0.8585 - loss: 0.3754 - val_accuracy: 0.9000 - val_loss: 0.2940
Epoch 7/10
280/280 ━━━━━━━━━━━━━━━━━━━━ 123s 439ms/step - accuracy: 0.8647 - loss: 0.3583 - val_accuracy: 0.8884 - val_loss: 0.3155
Epoch 8/10
280/280 ━━━━━━━━━━━━━━━━━━━━ 123s 438ms/step - accuracy: 0.8714 -

In [ ]:
model.save(
    "../models/efficientnet_brain_tumor.keras"
)